# 04 · Preprocessing & chunking

**Light cleaning only** — cue words (`decided`, `agreed`, `will`, `by Friday`, `need to`, …)
are always preserved because extraction depends on them. Chunking is token-aware,
keeps whole speaker turns, overlaps turns for context, and never truncates the meeting.

In [ ]:
print('[1/4] Cleaning keeps the cues')
import sys; sys.path.insert(0, '.')
from src.preprocessing import clean_text
raw = 'We um decided to ship by Friday, and we really really need to confirm.'
print('raw   :', raw)
print('clean :', clean_text(raw))
assert 'decided' in clean_text(raw) and 'by Friday' in clean_text(raw) and 'need to' in clean_text(raw)

In [ ]:
print('[2/4] Parse the synthetic sample transcript')
from src.config import load_config, resolve_path
from src.data_loader import load_transcript_file
from src.preprocessing import Turn, clean_text, merge_short_turns, turn_sentences
cfg = load_config()
m = load_transcript_file('data/samples/sample_meeting.txt')
turns = merge_short_turns([Turn(t['speaker'], clean_text(t['text']), t.get('start'), t.get('end'))
                           for t in m['turns'] if t['text']])
sents = turn_sentences(turns, m['meeting_id'])
print('turns:', len(turns), '| sentences:', len(sents), '| speakers:', m['participants'])
for t in turns[:4]:
    print(f"  [{t.span or 'no-ts'}] {t.speaker}: {t.text[:75]}")

In [ ]:
print('[3/4] Chunking: full coverage, token budget respected')
from src.chunking import chunk_turns, get_tokenizer
tok = get_tokenizer(cfg.summarization.model)
chunks = chunk_turns(turns, meeting_id=m['meeting_id'],
                     max_tokens=cfg.summarization.chunk_max_tokens,
                     overlap_turns=cfg.summarization.chunk_overlap_turns, tokenizer=tok)
print('chunks:', len(chunks))
for c in chunks:
    print(f'  {c.chunk_id}  tok={c.token_count:4d}  turns={c.n_turns}  '
          f'speakers={len(c.speakers)}  span={c.start_time}-{c.end_time}')
assert all(c.token_count <= cfg.summarization.chunk_max_tokens for c in chunks)

In [ ]:
print('[4/4] Save chunk metadata (resumable pipeline artifact)')
from src.chunking import save_chunks
from src.config import resolve_path
p = resolve_path(cfg, 'data_processed') / f"{m['meeting_id']}_chunks.json"
save_chunks(chunks, p)
print('->', p)